In [4]:
import time
import datetime as dt
import numpy as np
import pyvisa
from pathlib import Path

In [8]:
savepath = Path("C:/Users/IPMU/Desktop/ipmu_DAQ/Temperature")
savepath.mkdir(parents=True, exist_ok=True)

# look up instruments
rm = pyvisa.ResourceManager()
print("resource manager : ", rm)
rl = rm.list_resources()
print("List of resources: \n", rl)


resource manager :  Resource Manager of Visa Library at C:\WINDOWS\system32\visa32.dll
List of resources: 
 ('GPIB0::12::INSTR',)


In [9]:
for i,name in enumerate(rl):
    if "GPIB" in name:
        my_instrument = rm.open_resource(name)
        print("Resource[",i,"]:", name, "\n", my_instrument.query("*IDN?"))

Resource[ 0 ]: GPIB0::12::INSTR 
 LSCI,MODEL224,LSA17QI/OCD17QI/OCC17QI,1.1



In [ ]:
LS218 = [rm.open_resource(rl[2])]

In [ ]:
LS218_INPUT = 0
for i in range(7):
    command = "INPUT? " + str(i)
    if "1" in LS218[0].query(command):
        #print(i,LS218.query(command))
        LS218_INPUT += 1
        
        
filename = savepath / f"{dt.datetime.now().strftime('%Y%m%d%H%M%S')}_LS218.csv"
print(filename)

columnname = ['Abstime','Reltime','Ch1','Ch2','Ch3','Ch4','Ch5','Ch6','Ch7',] 
try:
    with open(filename, mode="a") as f:
        print(*columnname, sep=", ", file=f)
except:
    pass

REFRESHRATE = 1  # [sec] for frequency of daq
print(LS218_INPUT)


C:/Users/IPMU/Desktop/2025-steppingmotor-run6/Temperature/20260116230048_LS218.csv
8


In [7]:
def getTemp(self, ch=None):
    if ch == None:
        ch = "1"
    self.write("KRDG? %s" % ch)  # if ch=0, it mean readout all values
    T = float(self.read())
    return T

In [8]:
StarTime = time.time()
while True:
    
    CurrentAbsTime = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    CurrentRerTime = time.time() - StarTime
    arr = [CurrentAbsTime, CurrentRerTime]

    #===ls218, connector A: cryostat temperature monitoring===
    for i in range(1,LS218_INPUT+1):
        # skip 0 because 'KRDG? 0' means read all channel
        TE218 = getTemp(LS218[0],i)
        arr.append(TE218)

    print(arr)
    time.sleep(REFRESHRATE)

    try:
        
        with open(filename, mode="a") as f:
            print(*arr, sep=", ", file=f)

    except FileNotFoundError:

        print("FileNotFoundError... :'-(")
        filename = savepath / f"{dt.datetime.now().strftime('%Y%m%d%H%M%S')}_LS218.csv"
        with open(filename, mode="a") as f:
            print(*columnname, sep=", ", file=f)
            print(*arr, sep=", ", file=f)


['2026-01-16 23:00:50', 0.0, 298.33, 297.75, 298.81, 297.78, 298.6, 298.32, 296.98, 0.0]
['2026-01-16 23:00:51', 1.049018144607544, 298.33, 297.75, 298.82, 297.78, 298.6, 298.31, 296.98, 0.0]
['2026-01-16 23:00:52', 2.090994119644165, 298.33, 297.75, 298.82, 297.78, 298.6, 298.31, 296.98, 0.0]
['2026-01-16 23:00:53', 3.135389566421509, 298.33, 297.75, 298.81, 297.78, 298.6, 298.32, 296.98, 0.0]
['2026-01-16 23:00:54', 4.178641319274902, 298.33, 297.75, 298.82, 297.78, 298.6, 298.32, 296.98, 0.0]
['2026-01-16 23:00:55', 5.222919225692749, 298.33, 297.75, 298.81, 297.78, 298.6, 298.32, 296.98, 0.0]
['2026-01-16 23:00:56', 6.264960765838623, 298.33, 297.75, 298.82, 297.78, 298.6, 298.32, 296.98, 0.0]
['2026-01-16 23:00:57', 7.3093178272247314, 298.33, 297.75, 298.82, 297.78, 298.6, 298.32, 296.98, 0.0]
['2026-01-16 23:00:59', 8.35293173789978, 298.33, 297.75, 298.82, 297.78, 298.6, 298.31, 296.98, 0.0]
['2026-01-16 23:01:00', 9.393127202987671, 298.33, 297.75, 298.82, 297.78, 298.6, 298.3

KeyboardInterrupt: 